In [1]:
import polars as pl
import os

In [2]:
data_path = "raw_data/sales_pers.user_chunk_*.parquet"

users = pl.scan_parquet(data_path)

In [3]:
with pl.Config(tbl_rows=-1,tbl_cols=-1, tbl_width_chars=1000, fmt_str_lengths=1000):
    print(users.collect().head(100))

shape: (100, 18)
┌─────────────┬────────┬──────────┬───────────────────┬────────────┬────────────┬─────────────────────────┬────────────────────────────┬────────────────┬────────────────────────────┬────────────────────┬─────────────────────────┬─────────────────────────────────────────────────────────┬─────────────┬──────────────┬──────────────┬──────────────────────────────────────────────────────────────────┬────────────┐
│ customer_id ┆ gender ┆ location ┆ province          ┆ membership ┆ timestamp  ┆ created_date            ┆ updated_date               ┆ sync_status_id ┆ last_sync_date             ┆ sync_error_message ┆ region                  ┆ location_name                                           ┆ install_app ┆ install_date ┆ district     ┆ user_id                                                          ┆ is_deleted │
│ ---         ┆ ---    ┆ ---      ┆ ---               ┆ ---        ┆ ---        ┆ ---                     ┆ ---                        ┆ ---            ┆ ---  

In [4]:
users.collect_schema()

Schema([('customer_id', Int32),
        ('gender', String),
        ('location', Int32),
        ('province', String),
        ('membership', String),
        ('timestamp', Int64),
        ('created_date', Datetime(time_unit='us', time_zone=None)),
        ('updated_date', Datetime(time_unit='us', time_zone=None)),
        ('sync_status_id', Int32),
        ('last_sync_date', Datetime(time_unit='us', time_zone=None)),
        ('sync_error_message', String),
        ('region', String),
        ('location_name', String),
        ('install_app', String),
        ('install_date', Int64),
        ('district', String),
        ('user_id', String),
        ('is_deleted', Boolean)])

In [5]:
print(users.collect().shape)
users.collect().null_count()

(4573964, 18)


customer_id,gender,location,province,membership,timestamp,created_date,updated_date,sync_status_id,last_sync_date,sync_error_message,region,location_name,install_app,install_date,district,user_id,is_deleted
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,338285,338285,4573964,0,0,0,0,0,0,0


In [6]:
users = users.drop(["sync_error_message", "sync_status_id", "last_sync_date"])

In [7]:
print(users.collect().shape)
users.collect().null_count()

(4573964, 15)


customer_id,gender,location,province,membership,timestamp,created_date,updated_date,region,location_name,install_app,install_date,district,user_id,is_deleted
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [8]:
import polars.selectors as cs

In [9]:
user_string = users.select(cs.string())

In [10]:
user_string.collect().columns

['gender',
 'province',
 'membership',
 'region',
 'location_name',
 'install_app',
 'district',
 'user_id']

In [11]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("gender").is_not_null())
        .group_by("gender")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (3, 2)
┌────────┬─────────┐
│ gender ┆ count   │
│ ---    ┆ ---     │
│ str    ┆ u32     │
╞════════╪═════════╡
│ Nữ     ┆ 3424887 │
│ Nam    ┆ 1149069 │
│ Khác   ┆ 8       │
└────────┴─────────┘


In [12]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("province").is_not_null())
        .group_by("province")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (126, 2)
┌────────────────────────┬─────────┐
│ province               ┆ count   │
│ ---                    ┆ ---     │
│ str                    ┆ u32     │
╞════════════════════════╪═════════╡
│ Hồ Chí Minh            ┆ 1223333 │
│ Đồng Nai               ┆ 312131  │
│ Bình Dương             ┆ 287542  │
│ Hà Nội                 ┆ 201491  │
│ Đà Nẵng                ┆ 146596  │
│ Lâm Đồng               ┆ 134620  │
│ Long An                ┆ 133752  │
│ Bà Rịa - Vũng Tàu      ┆ 118101  │
│ Khánh Hòa              ┆ 112897  │
│ Tây Ninh               ┆ 112150  │
│ Đắk Lắk                ┆ 102728  │
│ An Giang               ┆ 87664   │
│ Tiền Giang             ┆ 86932   │
│ Cần Thơ                ┆ 83634   │
│ Kiên Giang             ┆ 82819   │
│ Bình Thuận             ┆ 81964   │
│ Đồng Tháp              ┆ 74808   │
│ Bình Phước             ┆ 71158   │
│ Vĩnh Long              ┆ 61105   │
│ Cà Mau                 ┆ 56627   │
│ Quảng Nam              ┆ 54252   │
│ Quảng Ngãi          

In [13]:
users = users.with_columns(
    province=(
        pl.col("province")
        .str.strip_chars()
        .str.replace(r"(?i)^(tỉnh|thành phố|tp\.?)\s+", "")
        .str.strip_chars()
    )
)
user_string = user_string.with_columns(
    province=(
        pl.col("province")
        .str.strip_chars()
        .str.replace(r"(?i)^(tỉnh|thành phố|tp\.?)\s+", "")
        .str.strip_chars()
    )
)
# Kiểm tra lại: số lượng tỉnh thành sẽ gom từ 126 về đúng 63 tỉnh thành chuẩn
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        users.group_by("province")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )
print("Số lượng tỉnh thành sau khi chuẩn hóa:", users.select(pl.col("province").n_unique()).collect().item())

shape: (63, 2)
┌───────────────────┬─────────┐
│ province          ┆ count   │
│ ---               ┆ ---     │
│ str               ┆ u32     │
╞═══════════════════╪═════════╡
│ Hồ Chí Minh       ┆ 1224319 │
│ Đồng Nai          ┆ 312380  │
│ Bình Dương        ┆ 287857  │
│ Hà Nội            ┆ 202879  │
│ Đà Nẵng           ┆ 146700  │
│ Lâm Đồng          ┆ 134658  │
│ Long An           ┆ 133896  │
│ Bà Rịa - Vũng Tàu ┆ 118216  │
│ Khánh Hòa         ┆ 113021  │
│ Tây Ninh          ┆ 112185  │
│ Đắk Lắk           ┆ 102988  │
│ An Giang          ┆ 87845   │
│ Tiền Giang        ┆ 87310   │
│ Cần Thơ           ┆ 83841   │
│ Kiên Giang        ┆ 82888   │
│ Bình Thuận        ┆ 82094   │
│ Đồng Tháp         ┆ 74865   │
│ Bình Phước        ┆ 71215   │
│ Vĩnh Long         ┆ 61172   │
│ Cà Mau            ┆ 56830   │
│ Quảng Ngãi        ┆ 54389   │
│ Quảng Nam         ┆ 54295   │
│ Bình Định         ┆ 53254   │
│ Thừa Thiên Huế    ┆ 52411   │
│ Hải Phòng         ┆ 51812   │
│ Sóc Trăng         ┆ 493

In [14]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("membership").is_not_null())
        .group_by("membership")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (3, 2)
┌────────────┬─────────┐
│ membership ┆ count   │
│ ---        ┆ ---     │
│ str        ┆ u32     │
╞════════════╪═════════╡
│ Standard   ┆ 4242946 │
│ Gold       ┆ 255173  │
│ Diamond    ┆ 75845   │
└────────────┴─────────┘


In [15]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("region").is_not_null())
        .group_by("region")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (8, 2)
┌───────────────────────────────┬─────────┐
│ region                        ┆ count   │
│ ---                           ┆ ---     │
│ str                           ┆ u32     │
╞═══════════════════════════════╪═════════╡
│ Đông Nam Bộ                   ┆ 2126066 │
│ Đồng bằng sông Cửu Long       ┆ 847970  │
│ Duyên hải Nam Trung Bộ        ┆ 577306  │
│ Đồng bằng sông Hồng           ┆ 361571  │
│ Tây Nguyên                    ┆ 333463  │
│ Bắc Trung Bộ                  ┆ 217869  │
│ Trung du và miền núi phía Bắc ┆ 105336  │
│ Duyên hải Bắc Bộ              ┆ 4383    │
└───────────────────────────────┴─────────┘


In [16]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("location_name").is_not_null())
        .group_by("location_name")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (995, 2)
┌─────────────────────────────────┬───────┐
│ location_name                   ┆ count │
│ ---                             ┆ ---   │
│ str                             ┆ u32   │
╞═════════════════════════════════╪═══════╡
│ DNA - 81 - 83 Nguyễn Văn Linh   ┆ 22383 │
│ HNI - Aeon Mall Hà Đông         ┆ 21746 │
│ HCM - 66 Nguyễn Du              ┆ 21509 │
│ HNI - 933 La Thành              ┆ 20835 │
│ HCM - 9 – 11 – 13 Nguyễn Trãi   ┆ 20563 │
│ DNA - 93-95 Lê Văn Hiến         ┆ 19558 │
│ HNI - 16B-4 Nguyễn Văn Lộc      ┆ 17538 │
│ HNI - 147K Đội Cấn              ┆ 16810 │
│ TVI - 62 Điện Biên Phủ          ┆ 16696 │
│ VLO - 47- 49 Trưng Nữ Vương     ┆ 16658 │
│ DON - 1-1/1 Hai Bà Trưng        ┆ 16517 │
│ CMA - 68 Trần Hưng Đạo          ┆ 16248 │
│ DNA - 842- 844 Tôn Đức Thắng    ┆ 15158 │
│ HCM - 223B Cống Quỳnh           ┆ 15018 │
│ TGI - 364-365 Nguyễn Huệ        ┆ 14009 │
│ TBI - Go! Thái Bình             ┆ 13994 │
│ HCM - 55B Phan Đăng Lưu         ┆ 13966 │
│ LDO - 4A - 4C 

In [17]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("install_app").is_not_null())
        .group_by("install_app")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (11, 2)
┌────────────────┬─────────┐
│ install_app    ┆ count   │
│ ---            ┆ ---     │
│ str            ┆ u32     │
╞════════════════╪═════════╡
│ In-Store       ┆ 3942000 │
│ SPE            ┆ 347908  │
│ iOS            ┆ 127882  │
│ Android        ┆ 91951   │
│ Web            ┆ 35876   │
│ CRM Partner    ┆ 15992   │
│ Call           ┆ 10584   │
│ Chat           ┆ 1727    │
│ Wholesale      ┆ 38      │
│ Không xác định ┆ 5       │
│ LZD            ┆ 1       │
└────────────────┴─────────┘


In [18]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        user_string
        .filter(pl.col("district").is_not_null())
        .group_by("district")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

shape: (1_339, 2)
┌───────────────────────────────┬────────┐
│ district                      ┆ count  │
│ ---                           ┆ ---    │
│ str                           ┆ u32    │
╞═══════════════════════════════╪════════╡
│ Thủ Đức                       ┆ 195601 │
│ Biên Hòa                      ┆ 125390 │
│ Bình Tân                      ┆ 105858 │
│ Tân Phú                       ┆ 102835 │
│ Bình Chánh                    ┆ 84427  │
│ 12                            ┆ 81984  │
│ Thuận An                      ┆ 71387  │
│ 1                             ┆ 65854  │
│ Gò Vấp                        ┆ 65727  │
│ 7                             ┆ 65561  │
│ Nha Trang                     ┆ 62789  │
│ Dĩ An                         ┆ 61637  │
│ Bình Thạnh                    ┆ 58133  │
│ Thủ Dầu Một                   ┆ 51715  │
│ Tân Bình                      ┆ 48774  │
│ Hóc Môn                       ┆ 48664  │
│ Đà Lạt                        ┆ 48346  │
│ Ninh Kiều                     ┆ 48

In [19]:
# Chuẩn hóa cột district
def clean_district_col(col_expr: pl.Expr) -> pl.Expr:
    return (
        col_expr.str.strip_chars()
        # 1. Bỏ tiền tố hành chính
        .str.replace(r"(?i)^(quận|huyện|thị xã|thành phố|tp\.?|tx\.?|q\.?|h\.?)\s+", "")
        .str.strip_chars()
        # 2. Chuẩn hóa các quận số thuần túy: '1' -> 'Quận 1', '12' -> 'Quận 12'
        .map_elements(
            lambda x: f"Quận {x}" if x and x.isdigit() else x,
            return_dtype=pl.String
        )
    )

users = users.with_columns(district=clean_district_col(pl.col("district")))
user_string = user_string.with_columns(district=clean_district_col(pl.col("district")))

# Kiểm tra lại top 20 quận huyện sau khi chuẩn hóa
with pl.Config(tbl_rows=20, tbl_cols=-1, tbl_width_chars=1000):
    print(
        users.group_by("district")
        .agg(pl.len().alias("count"))
        .sort("count", descending=True)
        .collect()
    )

print("Số lượng quận/huyện duy nhất sau khi gộp:", users.select(pl.col("district").n_unique()).collect().item())

shape: (706, 2)
┌─────────────────────┬────────┐
│ district            ┆ count  │
│ ---                 ┆ ---    │
│ str                 ┆ u32    │
╞═════════════════════╪════════╡
│ Thủ Đức             ┆ 196494 │
│ Biên Hòa            ┆ 125635 │
│ Bình Tân            ┆ 105992 │
│ Tân Phú             ┆ 102926 │
│ Bình Chánh          ┆ 84556  │
│ Quận 12             ┆ 82304  │
│ Thuận An            ┆ 71662  │
│ Quận 1              ┆ 65888  │
│ Gò Vấp              ┆ 65770  │
│ Quận 7              ┆ 65586  │
│ …                   ┆ …      │
│ Mù Cang Chải        ┆ 7      │
│ Nà Hang             ┆ 6      │
│ Phục Hoà            ┆ 3      │
│ Thị trấn Phú Riềng  ┆ 2      │
│ Thị trấn Long Thành ┆ 1      │
│ Tĩnh Gia            ┆ 1      │
│ Tp.Thủ Dầu Một      ┆ 1      │
│ Thông Nông          ┆ 1      │
│ Ia H Drai           ┆ 1      │
│ Tp.Thủ Đức          ┆ 1      │
└─────────────────────┴────────┘
Số lượng quận/huyện duy nhất sau khi gộp: 706


['gender',
 'province',
 'membership',
 'region',
 'location_name',
 'install_app',
 'district',
 'user_id']

In [20]:
users.collect_schema()

Schema([('customer_id', Int32),
        ('gender', String),
        ('location', Int32),
        ('province', String),
        ('membership', String),
        ('timestamp', Int64),
        ('created_date', Datetime(time_unit='us', time_zone=None)),
        ('updated_date', Datetime(time_unit='us', time_zone=None)),
        ('region', String),
        ('location_name', String),
        ('install_app', String),
        ('install_date', Int64),
        ('district', String),
        ('user_id', String),
        ('is_deleted', Boolean)])

In [22]:
with pl.Config(tbl_rows=-1, tbl_cols=-1, tbl_width_chars=1000):
    print(
        users.select(cs.numeric().is_null().sum()).collect()
    )

shape: (1, 4)
┌─────────────┬──────────┬───────────┬──────────────┐
│ customer_id ┆ location ┆ timestamp ┆ install_date │
│ ---         ┆ ---      ┆ ---       ┆ ---          │
│ u32         ┆ u32      ┆ u32       ┆ u32          │
╞═════════════╪══════════╪═══════════╪══════════════╡
│ 0           ┆ 0        ┆ 0         ┆ 0            │
└─────────────┴──────────┴───────────┴──────────────┘


In [26]:
import time
# 1. Thư mục đích
clean_dir = "clean_data"
os.makedirs(clean_dir, exist_ok=True)
output_path = os.path.join(clean_dir, "users_cleaned.parquet")
# 2. Loại bỏ cột timestamp bị trùng lặp với created_date (nếu còn)
if "timestamp" in users.collect_schema().names():
    users = users.drop(["timestamp"])
# 3. Xuất file bằng sink_parquet (Streaming cực nhanh và nhẹ RAM)
start_time = time.time()
users.sink_parquet(output_path, compression="zstd")
elapsed_time = time.time() - start_time
# 4. Kiểm tra và in thông số file đã lưu
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f" Xuất thành công users_clean sau: {elapsed_time:.2f} giây!")
print(f" Đường dẫn:   {output_path}")
print(f" Dung lượng:  {file_size_mb:.2f} MB")
print(f" Số dòng:     {users.select(pl.len()).collect().item():,} khách hàng")

 Xuất thành công users_clean sau: 2.41 giây!
 Đường dẫn:   clean_data\users_cleaned.parquet
 Dung lượng:  228.31 MB
 Số dòng:     4,573,964 khách hàng
